In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
print(pd.__version__)

2.3.3


In [2]:
CSV_PATH = "raw_wholesale_customers.csv"
df =pd.read_csv(CSV_PATH)
print(df.head())

   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


In [3]:
FEATURES = ["Fresh", "Milk", "Grocery",
            "Frozen", "Detergents_Paper", "Delicassen"]
X = df[FEATURES].copy()

In [4]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])

X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)

df[FEATURES] = X

In [5]:


print(f"Fresh  low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={X['Fresh'].min():.2f}  max={X['Fresh'].max():.2f}")
print(f"Milk   low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={X['Milk'].min():.2f}  max={X['Milk'].max():.2f}")
print( f"Grocery low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={X['Grocery'].min():.2f}  max={X['Grocery'].max():.2f}")
print(f"Frozen  low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={X['Frozen'].min():.2f}  max={X['Frozen'].max():.2f}")
print( f"Detergents_Paper  low={low_det:.2f}  high={high_det:.2f}  |  after cap min={X['Detergents_Paper'].min():.2f}  max={X['Detergents_Paper'].max():.2f}")
print(f"Delicassen   low={low_deli:.2f}  high={high_deli:.2f}  |  after cap min={X['Delicassen'].min():.2f}  max={X['Delicassen'].max():.2f}")


Fresh  low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk   low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen  low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen   low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25


Features after IQR cap

In [6]:
print(X.head())

     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


Scaling features

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaled shape:", X_scaled.shape)
print(X_scaled)


Scaled shape: (440, 6)
[[ 0.12857261  1.05158597  0.04926747 -0.95324427  0.09579175  0.06589216]
 [-0.42162716  1.08673463  0.35386453 -0.30973493  0.30651872  0.47075856]
 [-0.49064723  0.85804007  0.06793486 -0.04243744  0.38243489  2.46943983]
 ...
 [ 0.31112285  2.38267048  2.45460914 -0.86054235  2.39229863  0.55487464]
 [-0.10466425 -0.70014133 -0.7595007  -0.61070442 -0.75732904  0.7933576 ]
 [-0.84025742 -0.76473271 -0.71730938 -1.01518413 -0.65213577 -1.1228252 ]]


ELBOW METHOD

In [8]:
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")


k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, 

Fit K-Means with choosing k

In [9]:
k=3
kmeans = KMeans(n_clusters=k, n_init="auto", random_state=42)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)

print(df.head())

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        1  
1     1776.00        0  
2     3938.25        2  
3     1788.00        2  
4     3938.25        2  


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


Evaluating cluster

In [10]:
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
print("\n=== METRICS ===")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== METRICS ===
Silhouette Score : 0.340 (closer to +1 is better)
Davies–Bouldin   : 1.297 (lower is better)


In [11]:
# sil = silhouette_score(X_scaled, labels)
# dbi = davies_bouldin_score(X_scaled, labels)
# print("\n=== METRICS ===")
# print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
# print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")

In [12]:
# k=6
# kmeans = KMeans(n_clusters=k, n_init="auto", random_state=42)
# labels = kmeans.fit_predict(X_scaled)

# df["Cluster"] = labels.astype(int)

# print(df.head())

# sil = silhouette_score(X_scaled, labels)
# dbi = davies_bouldin_score(X_scaled, labels)
# print("\n=== METRICS ===")
# print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
# print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")

 Cluster centers 

In [14]:
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"
print(centers_df.round(2))

            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         5869.23  10142.25  16653.32  1355.63           7022.05     1482.60
1         9418.37   2685.47   3583.98  2052.33            942.49      750.71
2        22881.83   5870.35   6773.44  5057.43           1209.38     2452.04


Sanity checks 

In [15]:
sample_idx = [0, 1, 2]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]

print(sanity)

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  
0     1338.00        1  
1     1776.00        0  
2     3938.25        2  


In [17]:
OUT_PATH = "segmented_wholesale_customer.csv"
df.to_csv(OUT_PATH)
print("saved")

saved
